# Satellite Pose Estimation - Quick Start Guide

This notebook demonstrates the complete workflow for training and evaluating a satellite pose estimation model.

## Steps:
1. Load and explore the dataset
2. Create and visualize data samples
3. Train a DtF model
4. Evaluate on test set
5. Visualize results

In [ ]:
# Import libraries
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

from config import DATA_DIR, MODEL_DIR, RESULTS_DIR, TRAINING_CONFIG, MODEL_CONFIG
from data_loader import create_data_loaders, create_dummy_dataset
from models import create_model
from trainer import Trainer
from evaluation import evaluate_model

print("Libraries imported successfully!")
print(f"CUDA available: {torch.cuda.is_available()}")

## Step 1: Create Dummy Dataset (for testing)

If you don't have the full Kaggle dataset yet, we'll create dummy data to test the pipeline.

In [ ]:
# Create dummy dataset
print("Creating dummy dataset...")
create_dummy_dataset(
    DATA_DIR / "dataset" / "train",
    num_samples=100
)
create_dummy_dataset(
    DATA_DIR / "dataset" / "test",
    num_samples=20
)
print("✓ Dummy dataset created!")

## Step 2: Load and Explore Data

In [ ]:
# Create data loaders
print("Creating data loaders...")
train_loader, val_loader = create_data_loaders(
    DATA_DIR,
    batch_size=16,
    num_workers=2,
    sequence_type="random",
    augmentation=True,
)

print(f"✓ Created {len(train_loader)} training batches")
print(f"✓ Created {len(val_loader)} validation batches")

# Get a sample batch
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  Images: {batch['image'].shape}")
print(f"  Translation: {batch['translation'].shape}")
print(f"  Quaternion: {batch['quaternion'].shape}")

## Step 3: Visualize Sample Data

In [ ]:
# Visualize sample images and pose
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    if i < len(batch['image']):
        # Denormalize image
        img = batch['image'][i].permute(1, 2, 0).cpu().numpy()
        img = (img - img.min()) / (img.max() - img.min())
        
        ax.imshow(img)
        trans = batch['translation'][i].cpu().numpy()
        ax.set_title(f"Trans: [{trans[0]:.2f}, {trans[1]:.2f}, {trans[2]:.2f}]")
        ax.axis('off')

plt.tight_layout()
plt.savefig(RESULTS_DIR / "sample_images.png", dpi=100, bbox_inches='tight')
print("✓ Sample visualization saved")
plt.show()

## Step 4: Create and Train Model

In [ ]:
# Create model
print("Creating model...")
model = create_model(
    model_type="dtf",
    backbone="resnet50",
    pretrained=True,
    hidden_dims=MODEL_CONFIG["hidden_dims"],
    dropout=MODEL_CONFIG["dropout"],
)
print("✓ Model created!")

# Create trainer
device = "cuda" if torch.cuda.is_available() else "cpu"
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    learning_rate=TRAINING_CONFIG["learning_rate"],
    num_epochs=10,  # Short for demo
    optimizer_name=TRAINING_CONFIG["optimizer"],
    scheduler_name=TRAINING_CONFIG["scheduler"],
    save_dir=MODEL_DIR,
)
print(f"✓ Trainer created! Using device: {device}")

In [ ]:
# Train model (limited epochs for demo)
print("\nStarting training...")
history = trainer.train()
print("\n✓ Training completed!")

## Step 5: Plot Training Results

In [ ]:
# Plot training history
from visualization import plot_training_history

plot_training_history(
    history,
    output_path=RESULTS_DIR / "training_history.png"
)
print("✓ Training history plotted")
plt.show()

## Step 6: Evaluate Model

In [ ]:
# Evaluate on test set
print("Evaluating model...")
metrics = evaluate_model(
    model,
    val_loader,
    device=device,
    output_path=RESULTS_DIR,
)
print("\n✓ Evaluation completed!")
print(f"\nKey Metrics:")
print(f"  Translation MAE: {metrics.get('trans_mae', 0):.6f} m")
print(f"  Rotation MAE: {metrics.get('rot_mae', 0):.2f}°")

## Step 7: Visualize Evaluation Results

In [ ]:
# Load and visualize results
import json

with open(RESULTS_DIR / "evaluation_results.json") as f:
    results = json.load(f)

predictions = results["predictions"]
ground_truth = results["ground_truth"]

# Plot error distribution
from evaluation import PoseEvaluator
evaluator = PoseEvaluator()

trans_errors = []
rot_errors = []

for pred, gt in zip(predictions, ground_truth):
    from evaluation import translation_error, quaternion_distance
    pred_trans = np.array(pred["translation"])
    gt_trans = np.array(gt["translation"])
    pred_quat = np.array(pred["quaternion"])
    gt_quat = np.array(gt["quaternion"])
    
    trans_errors.append(translation_error(pred_trans, gt_trans))
    rot_errors.append(quaternion_distance(pred_quat, gt_quat))

from visualization import plot_error_distribution
plot_error_distribution(
    trans_errors,
    rot_errors,
    output_path=RESULTS_DIR / "error_distribution.png"
)
print("✓ Error distribution plotted")
plt.show()

## Summary

You've successfully completed the satellite pose estimation pipeline! 

### What we did:
1. ✓ Created a dataset (dummy data for demo)
2. ✓ Loaded and explored data
3. ✓ Visualized sample images
4. ✓ Created a DtF neural network model
5. ✓ Trained the model
6. ✓ Evaluated on test set
7. ✓ Visualized results

### Next steps:
- Download the full Kaggle dataset and retrain
- Experiment with different backbones (resnet101, efficientnet_b0)
- Adjust hyperparameters in config.py
- Compare DtF vs PnP vs Hybrid approaches
- Deploy model using inference.py

**Happy training!** 🚀